In [ ]:
'''
SUN DATABASE
white light images only
'''

In [ ]:
import cv2
import pandas as pd
import numpy as np
import tkinter as tk
from tkinter import filedialog
from datetime import datetime
import matplotlib.pyplot as plt
import os
from common import get_image_size, move_images
import shutil
import uuid

In [ ]:
# Open file dialog to select the spreadsheet file
spreadsheet_path = filedialog.askopenfilename(
    title="Select the Spreadsheet File",
    filetypes=[("Excel files", "*.xlsx")]
)
sun_data = pd.read_excel(spreadsheet_path)
# Display the first few rows of the dataframe to understand its structure
sun_data.head()

In [ ]:
# Replacing the header tags with their all-caps counterparts
sun_data.columns = [col.upper() for col in sun_data.columns]

In [ ]:
# Extracting distinct values of shape
distinct_shapes = sun_data['SHAPE'].unique()
distinct_shapes.tolist()

In [ ]:
'''
Shape data pre-processing
1. In cells where we find IIa(LST-NG), replacing that value with lla
2. Replacing values with the corresponding name
Ip	Pedunculated
Isp	Subpedunculated
Is	Sessile
IIa	Slightly elevated
'''

In [ ]:
# Replacing 'IIa(LST-NG)' with 'IIa' in the SHAPE column
sun_data['SHAPE'] = sun_data['SHAPE'].replace('IIa(LST-NG)', 'IIa')

# Verify if the replacement was successful
sun_data[sun_data['SHAPE'] == 'IIa(LST-NG)']

In [ ]:
sun_data.drop(columns=['PATHOLOGICAL DIAGNOSIS'], inplace=True)

In [ ]:
# Verifying the changes
sun_data.head()

In [ ]:
def extract_bounding_boxes(line):
    parts = line.strip().split(' ')
    filename = parts[0]
    bbox_strings = []

    # Iterate over the remaining parts of the line to get each bounding box
    for bbox_str in parts[1:]:
        # Split by comma and exclude the last element (class ID)
        bbox_values = bbox_str.split(',')[:-1]  # Excludes the class ID
        # Join the coordinates with commas and add to the list
        bbox_strings.append(','.join(bbox_values))

    # Join all bounding box strings; use a space only if there are multiple bounding boxes
    all_bboxes = ' '.join(bbox_strings) if len(bbox_strings) > 1 else ''.join(bbox_strings)
    return filename, all_bboxes

# Define the path where the case text files are located
txt_files_path = os.path.dirname(spreadsheet_path) + '/sundatabase_positive/annotation_txt'
# Initialize an empty list to store the new data rows
new_data_rows = []
# Iterate over each case in the original dataframe
for index, row in sun_data.iterrows():
    case_id = row['ID']
    shape = row['SHAPE']
    size = row['SIZE']
    location = row['LOCATION']
    
    # Construct the file path for the corresponding text file
    txt_file_path = os.path.join(txt_files_path, f'case{case_id}.txt')
    
    # Check if the text file exists
    if os.path.exists(txt_file_path):
        # Open and read the text file
        with open(txt_file_path, 'r') as file:
            for line in file:
                # We are going to exclude the last value referring to the class as it is 0 (polyp) in all the entries 
                
                filename, bounding_boxes = extract_bounding_boxes(line)
                             
                # Create a new row with combined information
                new_row = {
                    'FRAME': filename, 
                    'POLYP ID': case_id,
                    'LOCATION': location,
                    'SHAPE': shape, 
                    'SIZE': size,
                    'BOUNDING BOX': bounding_boxes
                }
                new_data_rows.append(new_row)

# Convert the list of new rows into a DataFrame
sun_data = pd.DataFrame(new_data_rows)
print(len(sun_data))
sun_data.head()

In [ ]:
base_folder = os.path.join(os.path.dirname(spreadsheet_path), 'sundatabase_positive')

# Initialize a list to store new rows with image properties
new_rows = []

# Iterate over each row in the Combined Data
for _, row in sun_data.iterrows():
    file_name = row['FRAME']
    case_id = row['POLYP ID']
    
    # Construct the full path of the image
    image_path = os.path.join(base_folder, f'case{case_id}', file_name)
    
    # Analyze the image and get its properties
    if os.path.exists(image_path):
        width, height = get_image_size(image_path)
        
        # Add the new data to the row
        new_row = row.to_dict()
        new_row.update({
            'WIDTH': width,
            'HEIGHT': height,
        })
        new_rows.append(new_row)

# Convert the list of new rows into a DataFrame
sun_data = pd.DataFrame(new_rows)
print(len(sun_data))
sun_data.head()

In [ ]:
# Number of unique polyps
unique_polyps = sun_data['POLYP ID'].nunique()

unique_polyps

In [ ]:
# Distribution of polyp sizes. Figures

In [ ]:
# Calculate the distribution of polyps by size for the training set
size_distribution = sun_data['SIZE'].value_counts().sort_index()
size_distribution

In [ ]:
# Distribution of polyp sizes. Histogram

In [ ]:
# Plot histograms for the distribution of polyps by size for both sets
plt.figure(figsize=(15, 6))
size_distribution.plot(kind='bar')
plt.title('Distribution of Polyps by Size')
plt.xlabel('Polyp Size')
plt.ylabel('Frequency')
plt.grid(axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of polyp sizes. Descriptive statistics

In [ ]:
# Extracting only the descriptive statistics for the polyp size in both sets
desc_stats_size = sun_data['SIZE'].describe()
desc_stats_size

In [ ]:
# Distribution of polyp sizes. Box plot

In [ ]:
# Generate a combined vertical box plot for both sets
plt.figure(figsize=(8, 6))
sun_data['SIZE'].plot(kind='box')
plt.title('Box Plot of Polyp Sizes')
plt.ylabel('Size')
plt.grid(True)
plt.show()

In [ ]:
#Distribution by number of polyps (bounding boxes)

In [ ]:
# Define a function to count the number of polyps based on bounding boxes
def count_polyps(bbox_str):
    # Split the bounding box string by spaces
    bbox_list = bbox_str.split(' ')
    # Return the number of bounding boxes, which corresponds to the number of polyps
    return len(bbox_list)

# Apply the function to the 'BOUNDING BOX' column
sun_data['NUM POLYP'] = sun_data['BOUNDING BOX'].apply(count_polyps)

# Calculate the distribution of the number of polyps
polyp_distribution = sun_data['NUM POLYP'].value_counts()

# Sort the distribution by the number of polyps
polyp_distribution_sorted = polyp_distribution.sort_index()

# Print the distribution
print(polyp_distribution_sorted)


In [ ]:
sun_data.drop(columns=['NUM POLYP'], inplace=True)

In [ ]:
# Polyp localization analysis. Descriptive statistics

In [ ]:
# Function to extract coordinates from bounding box string
def extract_coordinates(bbox_str):
    # Split the string by spaces to separate each bounding box
    bboxes = bbox_str.split(' ')
    # Initialize lists to store coordinates
    coordinates = {'min_x': [], 'min_y': [], 'max_x': [], 'max_y': []}
    # Process each bounding box
    for bbox in bboxes:
        if bbox:
            mx, my, Mx, My = map(int, bbox.split(','))
            coordinates['min_x'].append(mx)
            coordinates['min_y'].append(my)
            coordinates['max_x'].append(Mx)
            coordinates['max_y'].append(My)
    return coordinates

# Apply the function and store the results in new DataFrame columns
coordinate_data = sun_data['BOUNDING BOX'].apply(extract_coordinates).apply(pd.Series)

# Calculate descriptive statistics for each coordinate
stats = {key: pd.Series([coord for sublist in coordinate_data[key] for coord in sublist]).describe() 
         for key in ['min_x', 'min_y', 'max_x', 'max_y']}

# Convert the stats dictionary to a DataFrame for better presentation
stats_df = pd.DataFrame(stats)

# Print the descriptive statistics
print(stats_df)

In [ ]:
# We identify the entries with negative coordinate values
# Function to check for negative coordinates in a bounding box string
def has_negative_coordinates(bbox_str):
    # Split the string by spaces to separate each bounding box
    bboxes = bbox_str.split(' ')
    for bbox in bboxes:
        # Split the coordinates and convert them to integers
        coordinates = map(int, bbox.split(','))
        # Check if any coordinate is negative
        if any(coord < 0 for coord in coordinates):
            return True
    return False

# Apply the function to each bounding box and create a new column for flags
sun_data['NEGATIVE COORDS'] = sun_data['BOUNDING BOX'].apply(has_negative_coordinates)

# Filter the DataFrame to get only the rows with negative coordinates
entries_with_negative_coordinates = sun_data[sun_data['NEGATIVE COORDS']]
# Drop the temporary column 'Has Negative Coordinates'
sun_data.drop('NEGATIVE COORDS', axis=1, inplace=True)

print(entries_with_negative_coordinates)
sun_data.head()


In [ ]:
# Let's analyze visually what happens with images in the train set that have negative coordinates

In [ ]:
# Define the path where the case text files are located
image_directory = os.path.dirname(spreadsheet_path) + '/sundatabase_positive/'

# Function to draw bounding boxes on an image
def draw_bounding_boxes(row):
    # Construct the image path
    case_folder = f"case{row['POLYP ID']}"
    image_path = image_directory + case_folder + '/' + row['FRAME']
    image = cv2.imread(image_path)

    if image is not None:
        # Convert the image from BGR to RGB
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Normalize pixel values if necessary
        if image_rgb.max() > 1.0:
            image_rgb = image_rgb / 255.0

        # Draw each bounding box on the image
        bboxes = row['BOUNDING BOX'].split(' ')
        for bbox in bboxes:
            x1, y1, x2, y2 = map(int, bbox.split(','))
            # Note: Matplotlib expects (R, G, B) order of colors
            cv2.rectangle(image_rgb, (x1, y1), (x2, y2), (0, 1, 0), 2)  # Green box

        # Display the image in RGB format
        plt.imshow(image_rgb)
        plt.title(f"Image ID: {row['FRAME']}")
        plt.show()
    else:
        print(f"Image not found: {image_path}")

for index, row in entries_with_negative_coordinates.iterrows():
    # Draw bounding boxes on the image
    draw_bounding_boxes(row)


In [ ]:
# Replacing negative coordinate values with "0" seems to be safe 

In [ ]:
# Function to replace negative coordinates with zero
def replace_negative_coordinates(bbox_str):
    modified_bboxes = []
    bboxes = bbox_str.split(' ')
    for bbox in bboxes:
        coords = bbox.split(',')
        # Replace negative coordinates with zero
        modified_coords = [max(int(coord), 0) for coord in coords]
        modified_bbox = ','.join(map(str, modified_coords))
        modified_bboxes.append(modified_bbox)
    return ' '.join(modified_bboxes)

# Apply the function to each bounding box
sun_data['BOUNDING BOX'] = sun_data['BOUNDING BOX'].apply(replace_negative_coordinates)

# Double-checking we no longer have negative coords
sun_data['NEGATIVE COORDS'] = sun_data['BOUNDING BOX'].apply(has_negative_coordinates)
entries_with_negative_coordinates = sun_data[sun_data['NEGATIVE COORDS']]
# Drop the temporary column 'Has Negative Coordinates'
sun_data.drop('NEGATIVE COORDS', axis=1, inplace=True)
print(entries_with_negative_coordinates)
sun_data.head()

In [ ]:
# Are there bounding boxes that exceed the boundaries of the images? 

In [ ]:
# Function to check if bounding boxes exceed image boundaries
def check_bbox_boundaries(bbox_str, img_width, img_height):
    bboxes = bbox_str.split(' ')
    for bbox in bboxes:
        if bbox:
            x1, y1, x2, y2 = map(int, bbox.split(','))
            if x1 < 0 or y1 < 0 or x2 > img_width or y2 > img_height:
                return True
    return False

# List to store entries with bounding boxes that exceed boundaries
exceeding_entries = []

# Check each entry and store those that exceed boundaries
for index, row in sun_data.iterrows():
    if check_bbox_boundaries(row['BOUNDING BOX'], row['WIDTH'], row['HEIGHT']):
        exceeding_entries.append(row)

# Convert the list to a DataFrame (optional)
exceeding_entries_df = pd.DataFrame(exceeding_entries)

exceeding_entries_df


In [ ]:
# Let's analyze visually what happens with those images

In [ ]:
plot_limit = 5 
image_counter = 0

for index, row in exceeding_entries_df.iterrows():
    # Draw bounding boxes on the image
    draw_bounding_boxes(row)
    image_counter += 1
    if image_counter >= plot_limit:
        break

In [ ]:
# Function to adjust bounding boxes
def adjust_bounding_boxes(bbox_str, img_width, img_height):
    adjusted_bboxes = []
    bboxes = bbox_str.split(' ')
    for bbox in bboxes:
        if bbox:
            x1, y1, x2, y2 = map(int, bbox.split(','))
            # Adjust x_max and y_max if they exceed image boundaries
            x2 = min(x2, img_width)
            y2 = min(y2, img_height)
            adjusted_bbox = ','.join(map(str, [x1, y1, x2, y2]))
            adjusted_bboxes.append(adjusted_bbox)
    return ' '.join(adjusted_bboxes)

# Apply the function to each bounding box
for index, row in sun_data.iterrows():
    img_width = row['WIDTH']
    img_height = row['HEIGHT']
    sun_data.at[index, 'BOUNDING BOX'] = adjust_bounding_boxes(row['BOUNDING BOX'], img_width, img_height)


In [ ]:
# Polyp morphology analysis

In [ ]:
# Analyzing the distribution of SHAPE in the validation set
polyp_morphology_distribution = sun_data['SHAPE'].value_counts()
polyp_morphology_distribution

In [ ]:
pair_distribution = sun_data.groupby(['WIDTH', 'HEIGHT']).size()
pair_distribution

In [ ]:
sun_data.head()

In [ ]:
print(len(sun_data))

In [ ]:
# image base path
image_directory = os.path.dirname(spreadsheet_path) + '/sundatabase_positive/'
# destination path
destination_image_directory = os.path.dirname(spreadsheet_path) + '/img_all/'

# Create the base destination folder if it doesn't exist
os.makedirs(destination_image_directory, exist_ok=True)

for index, row in sun_data.iterrows():
    case_id = row['POLYP ID']
    frame_filename = row['FRAME']
    
    # Define the source path (modify as needed if there's a specific subfolder structure)
    source_path = os.path.join(image_directory, f"case{case_id}", frame_filename)
    
    destination_path = os.path.join(destination_image_directory, frame_filename)
    
    # Copy the file from source to destination
    try:
        shutil.copy(source_path, destination_path)
    except IOError as e:
        print(f"Could not copy {frame_filename}. Error: {e}")

In [ ]:
sun_data['IMAGE TYPE'] = 'WL'

In [ ]:
# Saving the filtered DataFrame to a spreadsheet
save_path = os.path.join(os.path.dirname(spreadsheet_path), 'annotations.xlsx')
sun_data.to_excel(save_path, index=False)

In [ ]:
# image base path
source_image_directory = os.path.dirname(spreadsheet_path) + '/img_all/'
# destination path
destination_image_directory = os.path.dirname(spreadsheet_path) + '/img/'

move_images(source_image_directory, destination_image_directory, sun_data)

In [ ]:
# Let's deal with the negative images now (images that have no polyps)

In [ ]:
# image base path
negative_image_directory = os.path.dirname(spreadsheet_path) + '/sundatabase_negative/'

# destination path
destination_image_directory = os.path.dirname(spreadsheet_path) + '/negative_img/'

# Check if the target directory exists, if not, create it
if not os.path.exists(destination_image_directory):
    os.makedirs(destination_image_directory)

# Loop through each subfolder in the source directory
for i in range(1, 10):  # From 1 to 9
    case_folder = os.path.join(negative_image_directory, f'case{i}')
    if os.path.exists(case_folder):
        files = os.listdir(case_folder)
        for file in files:
            source_file_path = os.path.join(case_folder, file)
            if os.path.isfile(source_file_path):  # Check if it's a file
                 _, extension = os.path.splitext(source_file_path)
                 new_filename = uuid.uuid4().hex + extension
                 target_img_path = os.path.join(destination_image_directory, new_filename)
                 shutil.copy(source_file_path, target_img_path)